In [ ]:
#libraries for data analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 

In [ ]:
#patient encounters
filepath = "../data/raw/patient_encounters_2023.csv"
df = pd.read_csv(filepath)

In [ ]:
#start of preprocessing.py help for patient encounters

In [ ]:
#1 convert all null values



# Replace '?' with NaN across the entire dataframe
df.replace('?', np.nan, inplace=True)





In [ ]:
#3 age encoding
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25,
    '[30-40)': 35, '[40-50)': 45, '[50-60)': 55,
    '[60-70)': 65, '[70-80)': 75, '[80-90)': 85,
    '[90-100)': 95
}

df['age_numeric'] = df['age'].map(age_map)



In [ ]:
#4: ICD-9 Diagnostic Codes
diag_cols = ['diag_1', 'diag_2', 'diag_3']

for col in diag_cols:
    # Clean to string for safe checks
    code_str = df[col].astype(str).str.strip()

    # Pull first 3 numeric digits when present
    code_num = pd.to_numeric(
        code_str.str.extract(r'^(\d{3})')[0],
        errors='coerce'
    )  

    # Build category column
    df[col + '_category'] = np.select(
        [
            df[col].isna(),                         # missing
            code_str.str.startswith('250'),                # diabetes
            code_num.between(390, 459),                    # circulatory
            code_num.between(460, 519),                    # respiratory
            code_num.between(520, 579),                    # digestive
            code_str.str.startswith(('V', 'E'))            # external
        ],
        [
            'missing',
            'diabetes',
            'circulatory',
            'respiratory',
            'digestive',
            'external'
        ],
        default='other'
    )



In [ ]:
#5a Creating the Readmission Target — BINARY
# Create binary target: 1 = readmitted (<30 or >30), 0 = not readmitted (NO)
df['readmission_binary'] = (df['readmitted'] != 'NO').astype(int)



In [ ]:
#5b Exclude Death/Hospice Discharges (CRITICAL)

"""
    CRITICAL: Exclude patients who died or were discharged to hospice.
    These patients cannot be "readmitted" — including them creates data leakage
    because their outcome is guaranteed to be "no readmission" for reasons
    unrelated to clinical care quality.

    Discharge disposition IDs to exclude:
      11 = Expired
      13 = Hospice / home
      14 = Hospice / medical facility
      19 = Expired at home (Medicaid only, hospice)
      20 = Expired in a medical facility (Medicaid only, hospice)
      21 = Expired, place unknown (Medicaid only, hospice)

    Run this BEFORE creating your readmission target variable.
"""
exclude_dispositions = [11, 13, 14, 19, 20, 21]

n_before = len(df)

df = df[
    ~df['discharge_disposition_id'].isin(exclude_dispositions)
]

n_after = len(df)


In [ ]:
#6: Medication Feature Engineering
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

# Only keep medication columns that actually exist
med_cols = [c for c in med_cols if c in df.columns]

# Count medications that changed (anything not "No")
df['n_meds_changed'] = df[med_cols].apply(
    lambda row: sum(1 for v in row if v != 'No'),
    axis=1
)

# Binary flag: any medication changed
df['any_med_changed'] = (df['n_meds_changed'] > 0).astype(int)

# Count medications increased
df['n_meds_increased'] = df[med_cols].apply(
    lambda row: sum(1 for v in row if v == 'Up'),
    axis=1
)



In [ ]:
#7 Clinical Complexity Score
df['complexity_score'] = (
    df['num_lab_procedures'].fillna(0) / 50 +
    df['num_procedures'].fillna(0) / 6 +
    df['num_medications'].fillna(0) / 20 +
    df['number_diagnoses'].fillna(0) / 9 + 
    df['number_emergency'].fillna(0) / 3 +
    df['number_inpatient'].fillna(0) / 5
)



In [ ]:
#end of preprocessing hints for patient encounters

In [ ]:
#additional data cleaning

#number of meds patient is on



med_cols = [
        'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
        'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
        'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
        'miglitol', 'troglitazone', 'tolazamide', 'examide',
        'citoglipton', 'insulin', 'glyburide-metformin',
        'glipizide-metformin', 'glimepiride-pioglitazone',
        'metformin-rosiglitazone', 'metformin-pioglitazone'
    ]



# Count meds that are NOT 'No'
df['num_active_meds'] = (df[med_cols] != 'No').sum(axis=1)





In [ ]:




#race, drop na and other, one hot encode others
df = df.dropna(subset=['race'])

df = df[df['race'] != 'Other']

df = pd.get_dummies(df, columns=['race'])



In [ ]:
#gender, dropped unknown and mapped male/female

df = df[df['gender'] != 'Unknown/Invalid']

df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})



In [ ]:
#age, replaced it with its average
df['age'] = df['age_numeric']
df = df.drop(columns=['age_numeric'])



In [ ]:
#weight, changed blank to 0 and 1 for everything else, change weight name
df['weight'] = (
    df['weight'].notna() & (df['weight'] != '')
).astype(int)

df = df.rename(columns={'weight': 'weight_checked'})



In [ ]:
#payer code, dropped
df = df.drop(columns=['payer_code'])



In [ ]:
#medical specialty dropped
df = df.drop(columns=['medical_specialty'])



In [ ]:
#replace diag_1,2,3 with diag1,2,3 category and , 
df['diag_1'] = df['diag_1_category']
df['diag_2'] = df['diag_2_category']
df['diag_3'] = df['diag_3_category']

#remove value 'missing'
df = df[
    ~(
        (df['diag_1'] == 'missing') |
        (df['diag_2'] == 'missing') |
        (df['diag_3'] == 'missing')
    )
]
# drop diag_1,2,3_category
df = df.drop(columns=['diag_1_category', 'diag_2_category', 'diag_3_category'])

#one hot encode diag 1,2,3
df = pd.get_dummies(
    df,
    columns=['diag_1', 'diag_2', 'diag_3'],
    drop_first=True
)


In [ ]:
#label encoding for drugs
mapping = {
    'No': 0,
    'Steady': 1,
    'Up': 2,
    'Down': 2
}


med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

for col in med_cols:
    if col in df.columns:
        df[col] = df[col].map(mapping)

#drop columns since they have a singular value of negative
df = df.drop(columns=['examide', 'citoglipton', 'troglitazone'])

#binary encode diabetesMed
df['diabetesMed'] = df['diabetesMed'].map({'Yes': 1, 'No': 0})

In [ ]:
#'change' to binary or true/false
df['change'] = df['change'].map({'Ch': 1, 'No': 0})

In [ ]:
#update new csv
df.to_csv("../data/processed/patient_encounters_2023_processed.csv", index=False)



In [ ]:
pd.set_option('display.max_rows', None)

df.dtypes.reset_index().rename(columns={
    'index': 'column',
    0: 'dtype'
})